### Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install unrar tree
!pip install ipecharts

### Data Extract

In [ ]:
!unrar x /content/drive/MyDrive/TUKL/data_online_line_width_alpha.rar /content/
!mv /content/data_online_line_width_alpha/ /content/hwd/

In [4]:
!tree -L 1 /content/hwd/

/content/hwd/
├── aux_cache
├── DATA_CARD.md
├── Dataset
├── main_1000.csv
├── main_1500.csv
├── main_2000.csv
├── main_2500.csv
├── main_3000.csv
├── main_3500.csv
├── main_5000.csv
├── main_6000.csv
├── test_cache
├── test_cache2
├── test_cache3
├── test.csv
├── test_extended.csv
├── test_leakproof.csv
├── test_leakproof_raw.csv
├── train_cache
├── train_cache2
├── train_cache3
├── train.csv
├── train_extended.csv
├── train_leakproof.csv
├── train_leakproof_raw.csv
├── val_cache
├── val_cache3
├── val.csv
├── val_extended.csv
├── val_leakproof.csv
└── val_leakproof_raw.csv

10 directories, 21 files


In [6]:
import pandas as pd

df = pd.read_csv('/content/hwd/main_1500.csv')
print(df.shape)
df.head(2)

(253, 7)


,id,cms,gender,age,csv,img,line
0,0,461136,f,21,Dataset/Data_1500/csv/csv_0000_0.csv,Dataset/Data_1500/img/img_0000_0.png,پڑھی گئیں کہ ٹرکوں کے ٹرک امدادی سامان سے بھرے
1,0,461136,f,21,Dataset/Data_1500/csv/csv_0000_1.csv,Dataset/Data_1500/img/img_0000_1.png,نہ دیتا تو آپؒ اس پر ناراضگی کا اظہار کرتے


In [26]:
data = pd.read_csv('/content/hwd/Dataset/Data_1500/csv/csv_0001_0.csv')
data.head()[['X cood.','Y cood.','Time']]

,X cood.,Y cood.,Time
0,15563,1088,0
1,15492,1087,2
2,15477,1085,4
3,15462,1080,6
4,15451,1060,8


In [27]:
print(len(data))
print(len(data[(data['Pressure'] != 0)]))

23104
12171


In [28]:
data.head(1).T

,0
X cood.,15563.0
Y cood.,1088.0
Pressure,0.0
Height,-64.0
X tilt,2500.0
Y tilt,2500.0
Thickness,0.0
Time,0.0
dx,0.0
dy,0.0


In [39]:
binned = pd.cut(data['pen_down'], bins=4)

# Count per bin
counts = binned.value_counts().sort_index()
print(counts)


pen_down
(-0.001, 0.25]    10933
(0.25, 0.5]           0
(0.5, 0.75]           0
(0.75, 1.0]       12171
Name: count, dtype: int64


### Tensor Prep

In [46]:
import glob
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# 1. Load and combine ALL main_*.csv files gracefully
all_csv_files = glob.glob('/content/hwd/main_*.csv')
df = pd.concat((pd.read_csv(f) for f in all_csv_files), ignore_index=True)

print(f"Loaded {len(all_csv_files)} files. Total dataset size: {len(df)} samples")

# 2. Dataset
class SimpleHWDataset(Dataset):
    def __init__(self, dataframe, base_dir='/content/hwd/', min_dist=30.0):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = base_dir
        self.min_dist = min_dist # Minimum raw pixel distance between consecutive points
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        csv_path = self.base_dir + self.df.loc[idx, 'csv']
        points = pd.read_csv(csv_path)
        
        # 1. Sort by Time to trace the chronological path
        points = points.sort_values('Time')
        
        # 2. Filter out "hover" points (keep only when pen is touching)
        # Assuming pen_down is 1 for down, 0 for up based on your distribution
        if 'pen_down' in points.columns:
            points = points[points['pen_down'] > 0.5]
            
        coords = points[['X cood.', 'Y cood.']].values.astype(np.float32)
        
        # 3. Spatial Downsampling (Remove consecutive points that are too close)
        if len(coords) > 1 and self.min_dist > 0:
            kept_indices = [0]
            last_pt = coords[0]
            for i in range(1, len(coords)):
                # Calculate Euclidean distance between current point and the last kept point
                dist = np.linalg.norm(coords[i] - last_pt)
                if dist >= self.min_dist:
                    kept_indices.append(i)
                    last_pt = coords[i]
            coords = coords[kept_indices]
        
        # 4. Normalize between 0 and 1
        if len(coords) > 0:
            pt_max = coords.max(axis=0)
            pt_min = coords.min(axis=0)
            range_vals = pt_max - pt_min + 1e-6
            coords = (coords - pt_min) / range_vals
        
        # 5. Shuffle inputs to simulate unordered OpenCV points
        shuffle_idx = np.random.permutation(len(coords))
        unordered_coords = coords[shuffle_idx]
        
        # 6. Create target sequence
        target_seq = np.argsort(shuffle_idx)
        
        return torch.tensor(unordered_coords), torch.tensor(target_seq)

# 3. Simple Collate Function (for batch padding)
def simple_collate(batch):
    coords, targets = zip(*batch)
    
    # Pad sequences with 0s for coords and -1 for target labels (to ignore in loss)
    coords_padded = pad_sequence(coords, batch_first=True, padding_value=0.0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=-1)
    
    return coords_padded, targets_padded

# 4. Ready to use!
dataset = SimpleHWDataset(df)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=simple_collate)

# 4. Split the data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 5. Create Datasets
train_dataset = SimpleHWDataset(train_df)
val_dataset = SimpleHWDataset(val_df)

# 6. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=simple_collate)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=simple_collate)

# Verify
x, y = next(iter(train_loader))
print("Batched Inputs shape (Batch, Seq_Len, X/Y):", x.shape)
print("Batched Targets shape (Batch, Seq_Len):", y.shape)

Loaded 8 files. Total dataset size: 2403 samples
Batched Inputs shape (Batch, Seq_Len, X/Y): torch.Size([16, 1169, 2])
Batched Targets shape (Batch, Seq_Len): torch.Size([16, 1169])


### Training

#### Pointer Network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PointerAttention(nn.Module):
    """Calculates attention scores over the encoder outputs, pointing to previous items."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_state, encoder_outputs, mask):
        # decoder_state: (B, H), encoder_outputs: (B, S, H)
        dec_expanded = decoder_state.unsqueeze(1) # (B, 1, H)
        
        # Energy: (B, S, H) -> (B, S, 1) -> (B, S)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(dec_expanded))
        scores = self.V(energy).squeeze(-1)
        
        # Mask out padded positions to prevent them from being pointed to
        scores = scores.masked_fill(~mask, float('-inf'))
        return scores

class PointerNet(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128):
        super().__init__()
        # Embed the X, Y coordinates
        self.embedding = nn.Linear(input_dim, hidden_dim)
        
        # Encoder: Reads the unordered set (BiLSTM is fine, treats it as a sequence but learns local features)
        self.encoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        
        # Decoder: Produces the sorted sequence autoregressively
        self.decoder_cell = nn.LSTMCell(hidden_dim * 2, hidden_dim * 2)
        
        # Attention points to encoder outputs
        self.attention = PointerAttention(hidden_dim * 2)
        
    def forward(self, x, targets=None, teacher_forcing_ratio=0.5):
        B, max_len, _ = x.size()
        
        # Create mask based on padded 0s. (Assumes true points are normalized > 0 or < 1, rarely exact 0,0)
        # For a more robust mask, you can pass lengths natively from the dataset
        mask = (x.sum(dim=-1) != 0) 
        
        embedded = self.embedding(x)
        # encoder_outputs shape: (B, max_len, hidden_dim * 2)
        encoder_outputs, (h_n, c_n) = self.encoder(embedded) 
        
        # Initialize decoder state with final encoder states
        dec_h = torch.cat([h_n[0], h_n[1]], dim=-1)
        dec_c = torch.cat([c_n[0], c_n[1]], dim=-1)
        
        # Decoder starts with a blank input
        dec_input = torch.zeros(B, encoder_outputs.size(-1), device=x.device)
        logits_seq = []
        
        for t in range(max_len):
            dec_h, dec_c = self.decoder_cell(dec_input, (dec_h, dec_c))
            
            # Predict which input point comes next
            logits = self.attention(dec_h, encoder_outputs, mask)
            logits_seq.append(logits)
            
            # Next input to decoder
            if targets is not None and torch.rand(1).item() < teacher_forcing_ratio:
                # Teacher Forcing: Feed the true target's embedding for the next step
                valid_targets = torch.clamp(targets[:, t], min=0) # Handle -1 padding safely
                dec_input = encoder_outputs[torch.arange(B), valid_targets]
            else:
                # Autoregressive: Feed the model's highest probability prediction
                pred_idx = logits.argmax(dim=-1)
                dec_input = encoder_outputs[torch.arange(B), pred_idx]
                
        return torch.stack(logits_seq, dim=1) # Shape: (B, max_len, max_len)

#### Training Loop

In [ ]:
from tqdm.auto import tqdm

def train_pointer_net(model, train_loader, val_loader, epochs=10, lr=1e-3, device='cuda'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Ignore index -1 skips our sequence paddings automatically!
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # --- TRAINING ---
        model.train()
        train_loss = 0
        
        # tqdm for modern progress bar
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for x, y in train_pbar:
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            # Start with high teacher forcing, gradually decay it if you want
            logits = model(x, targets=y, teacher_forcing_ratio=0.5) 
            
            # Flatten to compute loss: (B*S, S) vs (B*S)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_pbar.set_postfix(loss=loss.item())
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- VALIDATION ---
        model.eval()
        val_loss, correct, total = 0, 0, 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for x, y in val_pbar:
                x, y = x.to(device), y.to(device)
                
                # Turn off teacher forcing for pure validation
                logits = model(x, targets=None, teacher_forcing_ratio=0.0)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                val_loss += loss.item()
                
                # Calculate accuracy (excluding padding)
                preds = logits.argmax(dim=-1)
                mask = (y != -1)
                correct += (preds[mask] == y[mask]).sum().item()
                total += mask.sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total if total > 0 else 0
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
    return history

## Visualization

In [ ]:
import matplotlib.pyplot as plt

def visualize_prediction(model, dataset, idx=0, device='cuda'):
    model.eval()
    model.to(device)
    
    # Grab a single sample (unbatched originally natively from dataset)
    x, y_true = dataset[idx]
    
    # Push to device and add a batch dimension: (1, S, 2)
    x_batch = x.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(x_batch, teacher_forcing_ratio=0.0)
        # Get predicted sequence of indices
        y_pred = logits[0].argmax(dim=-1).cpu() 
        
    x_coords = x.cpu().numpy()
    
    # Order points using true vs predicted indices
    # We only care about the valid points, not the padded -1s
    valid_len = (y_true != -1).sum().item()
    
    true_order = x_coords[y_true[:valid_len].numpy()]
    
    # The models autoregressive nature might repeat points if it gets confused, 
    # clip length conceptually for visual comparison.
    pred_order = x_coords[y_pred[:valid_len].numpy()]
    
    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.set_title("Ground Truth Path")
    ax1.plot(true_order[:, 0], true_order[:, 1], marker='o', markersize=3, color='blue', alpha=0.6)
    ax1.invert_yaxis() # Handwriting origin is usually top-left
    
    ax2.set_title("Model Prediction Path")
    ax2.plot(pred_order[:, 0], pred_order[:, 1], marker='x', markersize=3, color='red', alpha=0.6)
    ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.show()